# Revisi Jurnal — B1 (train-time augmentation) & B2 (multi-split)

Runner Colab untuk dua eksperimen yang menjawab keberatan reviewer terbesar.

| | Pertanyaan | Perkiraan waktu (T4) |
|---|---|---|
| **B1** | Collapse TTA di ClinTox itu efek imbalance, atau semata distribution shift karena model dilatih kanonik lalu diuji SMILES acak? | ~1 jam |
| **B2** | Apakah temuan TTA bertahan di scaffold split lain? (split utama cuma punya 9-10 molekul minoritas) | ~2-3 jam |

**Aman terhadap hasil lama:** semua artefak baru memakai nama model terpisah
(`chemberta_aug`, `chemberta_s1`, `chemberta_s2`) dan file split terpisah
(`{dataset}_split_s{k}.json`). Tidak satu pun angka paper yang tertimpa.

Jalankan sel berurutan. B1 dan B2 independen — boleh salah satu saja.


## 1. Cek GPU + mount Drive

In [ ]:
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'TIDAK ADA -> Runtime > Change runtime type > GPU')
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo (KODE)

In [ ]:
REPO = 'https://github.com/belvahector-ship-it/pharm_.git'
if not os.path.exists('/content/pharm_'):
    !git clone -q {REPO} /content/pharm_
%cd /content/pharm_
!git pull -q
print('repo siap')

## 3. Dependency — versi DIPIN ke environment run asli

In [ ]:
!pip install -q 'rdkit==2026.3.3' 'transformers==5.0.0' 'tokenizers==0.22.2' 'chemprop==2.2.4'
print('install selesai (versi dipin)')

## 4. Ekstrak `hasil_outputs.zip` dari Drive

Diperlukan agar model **base** (`chemberta` / `chemberta_tta`) tersedia sebagai pembanding
di laporan B1. Kalau path zip Anda berbeda, edit `ZIP_PATH`.

B2 tidak butuh sel ini (melatih dari nol di split baru), tapi menjalankannya tidak merugikan.

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/pharm_/hasil_outputs.zip'
import zipfile, glob, shutil
if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('/content/_unzip')
    moved = 0
    for src, dst in (('*.pt', 'outputs/checkpoints'), ('*.npy', 'outputs/predictions')):
        os.makedirs(dst, exist_ok=True)
        for f in glob.glob(f'/content/_unzip/**/{src}', recursive=True):
            shutil.copy(f, os.path.join(dst, os.path.basename(f))); moved += 1
    print(f'{moved} file dipindahkan')
else:
    print(f'ZIP tidak ditemukan di {ZIP_PATH} -- B1 tetap jalan, tapi kolom base_* akan NaN.')

## 5. B1 — Train-time SMILES augmentation (ClinTox)

Melatih `chemberta_aug`: training set diperbanyak dgn enumerasi SMILES (canonical + 3 acak),
**label direplikasi proporsional sehingga rasio kelas tidak berubah**. Validation tetap kanonik.

Ganti `--epochs 3` bila ingin menyamakan jumlah gradient step dgn model base (4x data -> 1 epoch = 4x step).
Tambah `bbbp bace` di `--datasets` bila ingin cakupan penuh (menambah ~1 jam).

In [ ]:
!python scripts/16_train_augmented.py --datasets clintox --seeds 0 1 2 3 4 5 6 7 8 9

### 5b. Baca hasil B1

In [ ]:
import pandas as pd
pd.set_option('display.width', 220, 'display.max_columns', 50)
s = pd.read_csv('outputs/results/train_augment/train_augment_summary.csv')
display(s)
print(open('outputs/results/train_augment/TRAIN_AUGMENT_REPORT.md', encoding='utf8').read())

## 6. B2 — Replikasi di 2 scaffold split tambahan

Melatih ChemBERTa dari nol di split 1 & 2 (protokol `scaffold_balanced`, Chemprop/Yang et al. 2019),
lalu menjalankan TTA + binary gate + instance gate di masing-masing.

Sudah diverifikasi lokal: test fold split baru hanya beririsan **9-14%** dgn split utama, jadi
molekul minoritasnya benar-benar berbeda (ClinTox: 9 -> 11 -> 16 molekul).

5 seed sudah cukup untuk replikasi. Naikkan ke 10 kalau anggaran GPU memungkinkan.

In [ ]:
!python scripts/17_multisplit_tta.py --split_seeds 1 2 --datasets clintox bbbp bace --seeds 0 1 2 3 4

### 6b. Baca hasil B2

In [ ]:
import pandas as pd
f = pd.read_csv('outputs/results/multisplit/multisplit_folds.csv')
d = pd.read_csv('outputs/results/multisplit/multisplit_per_seed.csv')
display(f)
display(d.groupby(['dataset','split_seed'])[
    ['auc_solo','auc_tta','auc_binary_gate','auc_instance_gate']].mean().round(4))
print(open('outputs/results/multisplit/MULTISPLIT_REPORT.md', encoding='utf8').read())

## 7. Simpan hasil ke Drive

In [ ]:
OUT_DRIVE = '/content/drive/MyDrive/pharm_revisi_jurnal'
os.makedirs(OUT_DRIVE, exist_ok=True)
import glob, shutil
n = 0
for sub in ('train_augment', 'multisplit'):
    src = f'outputs/results/{sub}'
    if not os.path.isdir(src):
        continue
    dst = os.path.join(OUT_DRIVE, sub); os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f'{src}/*'):
        shutil.copy(f, dst); n += 1
# checkpoint & prediksi baru saja (jangan timpa arsip lama)
ck = os.path.join(OUT_DRIVE, 'checkpoints'); os.makedirs(ck, exist_ok=True)
for pat in ('chemberta_aug_*', 'chemberta_s1_*', 'chemberta_s2_*'):
    for f in glob.glob(f'outputs/checkpoints/{pat}') + glob.glob(f'outputs/predictions/{pat}'):
        shutil.copy(f, ck); n += 1
for f in glob.glob('data/splits/*_s[12].json'):
    shutil.copy(f, OUT_DRIVE); n += 1
print(f'{n} file tersimpan ke {OUT_DRIVE}')